# Step 08.2 — Vanilla RNN Equation：从递推直觉到可学习的 Hidden State

上一节我们得到了抽象的递推 `h_t = f_θ(x_t, h_(t-1))`，但还不知道函数 `f_θ` 怎样实现。

本节会从已经学过的 **Linear、Bias、Activation、Shape** 一步步推到真正的经典 RNN：

$$
h_t=\tanh(W_x x_t+W_h h_{t-1}+b)
$$

五部分：① 拆解 RNN 的两条信息通路；② 严格推导全部 weight 的 shape；③ 手算连续两个时间步；④ Apple MLX 对照验证并理解 tanh；⑤ 权重共享、模型输出与下一步时间循环的衔接。

这节只写单步 `rnn_step` 并显式调用两次；完整 `for t in range(T)` 和 `nn.Module` 版本分别留给 8.3、8.4。


## 1. RNN 不是神秘新运算：两次线性映射，加法，再经过 tanh

先把内部的 pre-activation（激活前向量）单独写出来：

$$
a_t = W_xx_t + W_hh_{t-1} + b
$$

再做逐元素非线性：

$$
h_t = \tanh(a_t).
$$

其中有两条不同的信息通路：**input path** `W_x x_t` 将新输入映射到 H 维；**recurrent path** `W_h h_(t-1)` 将已有历史摘要映射到新的 H 维。两者合并后才产生当前的新记忆。

与 position-wise MLP 最大区别：在时刻 t，RNN 计算明确依赖更早计算留下的 `h_(t-1)`，而不仅是 `x_t` 本身。

这里的 `h_t` 是隐藏状态，不是下一个 token 的分类概率；如需要预测 token，后面还要接输出层。


### 这张计算图里的箭头代表信息依赖

箭头不表示 GPU 核心分配；权重 `W_x,W_h` 是所有时间步共享的参数。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3))
ax.set_xlim(0, 10)
ax.set_ylim(-0.3, 3.0)
ax.axis('off')

def box(x, y, label):
    ax.text(x, y, label, ha='center', va='center',
            bbox={'boxstyle':'round,pad=0.36', 'fill':False})

def arrow(x1, y1, x2, y2):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops={'arrowstyle':'->', 'lw':1.4})

box(0.8, 2.3, 'x_t')
box(3.3, 2.3, 'W_x x_t')
box(0.8, 0.5, 'h_(t-1)')
box(3.3, 0.5, 'W_h h_(t-1)')
box(5.5, 1.4, 'sum + bias')
box(7.5, 1.4, 'tanh')
box(9.0, 1.4, 'h_t')
arrow(1.35, 2.3, 2.55, 2.3)
arrow(1.35, 0.5, 2.4, 0.5)
arrow(4.05, 2.3, 4.95, 1.7)
arrow(4.05, 0.5, 4.95, 1.1)
arrow(6.2, 1.4, 6.86, 1.4)
arrow(8.1, 1.4, 8.5, 1.4)
ax.set_title('One vanilla RNN step: input path + history path')
plt.show()


## 2. 严格推导 Shape：为什么 `W_x=(H,C)`，`W_h=(H,H)`？

先统一符号：`C` 是每个时间步输入 feature width（例如 token embedding width），`H` 是 hidden state width，它们**不要求相等**。时刻索引 t 不是向量宽度。

本节用 `t=0` 表示第一个 token，因此初始状态写成 `h_(-1)=0`。有些教材从 t=1 计数，会写 `h_0=0`；那只是时间编号不同，不是两种 RNN。

### 数学的列向量约定

$$x_t\in\mathbb{R}^{C\times1},\quad h_{t-1}\in\mathbb{R}^{H\times1}$$

必须让三个加法项都有形状 `(H,1)`：

$$W_x\in\mathbb{R}^{H\times C},\qquad W_h\in\mathbb{R}^{H\times H},\qquad b\in\mathbb{R}^{H\times1}$$

于是

$$ (H,C)@(C,1)=(H,1),\qquad (H,H)@(H,1)=(H,1). $$

相加、tanh 以后 `h_t` 仍为 `(H,1)`。

### Apple MLX 使用 row-vector / batch convention

在真实代码里，单个 `x_t` 写作 `(C,)`，单个 `h_prev` 写作 `(H,)`；如果加 Batch，则

```text
x_t.shape   = (B,C)
h_prev      = (B,H)
W_x         = (H,C)
W_h         = (H,H)
b           = (H,)

x_t @ W_x.T      -> (B,H)
h_prev @ W_h.T   -> (B,H)
bias broadcasting-> (B,H)
mx.tanh(...)     -> (B,H)
```

这与 Step 4.1 `nn.Linear(C,H)` 的 weight 存储方式完全一致。

具体例子：`C=3,H=2,B=4` 时 `x_t=(4,3)`、`W_x=(2,3)`、`W_h=(2,2)`、`h_prev=(4,2)`，RNN 输出 `(4,2)`。


In [ ]:
import mlx.core as mx

def rnn_step(x_t, h_prev, W_x, W_h, b):
    # 单个 x_t=(C,), h_prev=(H,)；带 B 的情况也通用
    input_term = x_t @ W_x.T
    history_term = h_prev @ W_h.T
    preactivation = input_term + history_term + b
    return mx.tanh(preactivation)

C_demo, H_demo, B_demo = 3, 2, 4
x_batch = mx.ones((B_demo, C_demo))
h_previous_batch = mx.zeros((B_demo, H_demo))
W_x_demo = mx.ones((H_demo, C_demo))
W_h_demo = mx.zeros((H_demo, H_demo))
b_demo = mx.zeros((H_demo,))

h_batch = rnn_step(x_batch, h_previous_batch, W_x_demo, W_h_demo, b_demo)
mx.eval(h_batch)
print('x_batch         :', x_batch.shape)
print('h_previous_batch:', h_previous_batch.shape)
print('W_x / W_h / b  :', W_x_demo.shape, W_h_demo.shape, b_demo.shape)
print('h_batch         :', h_batch.shape)


### 为什么此时不必规定 `H=C`？

输入特征和历史记忆各自承担不同角色，线性映射负责把它们转换到统一的 H 维空间。RNN 可选不同的 hidden width；但与我们的 Decoder-Only MiniGPT 不同，RNN 的 H 维并非必须等于 Transformer 的 C 维。这里仅是学习一个历史模型，不修改 MiniGPT 主架构。


## 3. 连续两个时间步手算：历史怎样真正进入新状态？

选最透明的 `C=H=2`，使用**人工指定、尚未训练的参数**：

$$
W_x=\begin{bmatrix}1&0\\0&1\end{bmatrix},\quad
W_h=\begin{bmatrix}0.5&0\\0&0.5\end{bmatrix},\quad
b=\begin{bmatrix}0\\0\end{bmatrix}.
$$

注意这里的 `W_h` 是对角矩阵，只是方便手算；真正训练得到的 `W_h` 一般可以包含非零的非对角项，让不同 hidden features 彼此混合。

初始状态设为 `h_(-1) = [0,0]`。

### 时间步 t=0：读入 x0 = [1,0]

$$ a_0=W_xx_0+W_hh_{-1}+b=[1,0]^T. $$

tanh 作用在每一个分量上：

$$ h_0=[\tanh(1),\tanh(0)]^T\approx[0.7616,0]^T. $$

### 时间步 t=1：读入 x1 = [0,1]

新输入贡献 `W_x x_1=[0,1]`；历史贡献 `W_h h_0=[0.3808,0]`。

$$ a_1=[0,1]^T+[0.3808,0]^T=[0.3808,1]^T. $$

$$ h_1\approx[0.3634,0.7616]^T. $$

**关键观察**：当前输入 `x_1` 第一维为 0，但 `h_1` 第一维约为 0.3634——它来自上一步 `x_0` 的影响。RNN 产生了真实的历史依赖。

请注意 h 的数值不是“上一状态直接复制过来”：历史先过 `W_h`，再与新输入相加，最后受到非线性 tanh 调节。


In [ ]:
# 和上一格 rnn_step 定义完全对应的手算复现
W_x = mx.array([[1.0, 0.0], [0.0, 1.0]])
W_h = mx.array([[0.5, 0.0], [0.0, 0.5]])
bias = mx.array([0.0, 0.0])

h_minus_1 = mx.zeros((2,))
x0 = mx.array([1.0, 0.0])
x1 = mx.array([0.0, 1.0])

h0 = rnn_step(x0, h_minus_1, W_x, W_h, bias)
h1 = rnn_step(x1, h0, W_x, W_h, bias)
mx.eval(h0, h1)

print('h0 (expected ~[0.7616,0.0000]):', h0)
print('h1 (expected ~[0.3634,0.7616]):', h1)
expected_h1 = mx.array([0.36339948, 0.76159416])
print('manual result verified:', bool(mx.allclose(h1, expected_h1, atol=1e-4)))


### 用控制变量验证 Memory，而不只是看两次输出

保持 `x1` 不变，但把之前的状态改回零，看看当前结果是否还一样。

如果相同 x1 因历史 hprev 不同而得到不同 h，说明 RNN 的信息路径确实依赖历史。

**思考题：**如果把 `W_h` 设成全零矩阵，RNN 的输出还会记得上一步吗？为什么？


In [ ]:
h1_no_history = rnn_step(x1, mx.zeros((2,)), W_x, W_h, bias)
h0_reverse = rnn_step(x1, h_minus_1, W_x, W_h, bias)
h1_reverse = rnn_step(x0, h0_reverse, W_x, W_h, bias)
mx.eval(h1_no_history, h0_reverse, h1_reverse)

print('same x1, but without history:', h1_no_history)
print('same x1, with history       :', h1)
print('same current input, different states?',
      not bool(mx.allclose(h1, h1_no_history)))
print('reverse order final state:', h1_reverse)


## 4. 为什么选择 tanh？用曲线观察有界性与非线性

我们在 Step 4 已经知道：如果只有线性操作，层叠以后依然可合并为更简单的线性/仿射映射。RNN 需要非线性，才能建立更丰富的序列状态更新。

经典 Vanilla RNN 常用 `tanh`：

$$\tanh(z)=\frac{e^z-e^{-z}}{e^z+e^{-z}}.$$

你暂时只要理解这三个特性：

`tanh(0)=0`；当 z 很正时接近 +1、很负时接近 -1；输入每一维的任何实数值，输出都在 `(-1,1)`。

相比 ReLU，tanh 允许隐藏状态有正负数，而且单次输出有界。这不代表整个训练过程一定稳定，更不能保证不会梯度消失或无法学习长期依赖。

下面把 `y=z` 与 `y=tanh(z)` 画在同一个坐标系里，看哪里出现饱和。


In [ ]:
import math
zs = [(-4.0 + i*0.04) for i in range(201)]
tanh_values = [math.tanh(z) for z in zs]

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(zs, zs, label='identity: y=z', linestyle='--')
ax.plot(zs, tanh_values, label='tanh(z)')
ax.axhline(1, linewidth=0.8, linestyle=':')
ax.axhline(-1, linewidth=0.8, linestyle=':')
ax.set_ylim(-1.7,1.7)
ax.set_xlabel('pre-activation z')
ax.set_ylabel('activation output')
ax.set_title('Vanilla RNN nonlinearity: tanh is bounded')
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()


绘图技巧：`ax.plot()` 可以画连续函数趋势；`linestyle='--'` 与 `':'` 让参考线有区别；`ax.set_ylim()` 用来聚焦关注范围，不改变计算本身。

### 需要提前知道的限制（详细推导放到 Step 9）

当输入 z 的绝对值很大时，tanh 曲线接近平坦，意味着局部变化率很小。未来在 BPTT 中，很多时间步的局部导数连乘可能导致 **vanishing gradient**。不要把 tanh 的输出有界误认为它能解决任意长距离记忆。


## 5. 同一套参数跨时间使用；h_t 是状态，不等于最终预测

观察我们刚写的两次调用：

```python
h0 = rnn_step(x0, h_minus_1, W_x, W_h, bias)
h1 = rnn_step(x1, h0,        W_x, W_h, bias)
```

两次调用使用的是**同一组** `W_x,W_h,b`，不是每个 t 新建一套参数。

所以对于 input width C、hidden width H，这个 cell 的 trainable parameter 数（包含 bias）是：

$$P=HC+H^2+H$$

例如 `C=3,H=2`，就是 `2*3+2*2+2=12` 个参数，无论处理 5、50 还是 500 个时间步，**这套权重自身的数量不因 T 而增加**。

但是时间步更多，forward 的计算量、需要保留的中间状态以及训练时的 BPTT 图都可能增长；权重参数数目不变不等于运行成本不变。

### h_t 能不能直接当 next-token prediction？

不能简单画等号。这里的 `h_t` 是 `(H,)` 或 `(B,H)` 的**隐状态**；如果输出要对 `Vocab` 中每个 token 打分，通常还需要额外输出变换，例如 `logits_t = h_t @ W_out.T + b_out`，其中 `W_out.shape=(V,H)`。

和我们的 MiniGPT 类似，内部 hidden vector 与最终 `(B,T,V)` 的 token logits 是两种不同空间。

### 和 Step 7 的关系

RNN 在每个时间步可直接接收 Step 7 产生的 token embedding `x_t=(B,C)`。因为 recurrence 本身已有顺序敏感性，经典 RNN 不必像纯 content-only self-attention 那样依赖额外 absolute position embedding 才获得时间方向。

如果收集每个时间步的 hidden states，未来我们希望得到 `(B,T,H)`。至于如何从 `(B,T,C)` 依次取 `X[:,t,:]`、正确处理 Batch 并写出完整循环，我们留到 **8.3**。

### 8.2 最终自检

请确认你能解释：`W_x=(H,C)`、`W_h=(H,H)` 为什么必须这样；`h0` 怎么影响 `h1`；为什么 `tanh` 不改变 shape；为什么整条序列复用同一套权重；为什么 RNN hidden state 不等于语言模型分类 logits。

**下一节 8.3 — Time Loop Implementation**：真正执行 `for t in range(T)`、读取 `X[:,t,:]`，把一条变长 sequence 逐步处理并收集所有状态。
